[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/15_mlp.ipynb)

# 🟠 Medium: SwiGLU MLP

Implement the **SwiGLU MLP** (feed-forward network) used in modern LLMs like LLaMA.

$$\text{SwiGLU}(x) = \text{down\_proj}\big(\text{SiLU}(\text{gate\_proj}(x)) \odot \text{up\_proj}(x)\big)$$

where $\text{SiLU}(x) = x \cdot \sigma(x)$

### Signature
```python
class SwiGLUMLP(nn.Module):
    def __init__(self, d_model: int, d_ff: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.gate_proj`: `nn.Linear(d_model, d_ff)`
- `self.up_proj`: `nn.Linear(d_model, d_ff)`
- `self.down_proj`: `nn.Linear(d_ff, d_model)`
- Activation: **SiLU** (a.k.a. Swish) — `F.silu` or implement as `x * torch.sigmoid(x)`

### Why SwiGLU?
Unlike the classic `Linear → ReLU/GELU → Linear` FFN, SwiGLU uses a **gating mechanism**:
the gate projection controls information flow, while the up projection provides the content.
This consistently outperforms standard FFNs in practice (PaLM, LLaMA, Mistral all use it).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 828.9 kB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [7]:
# 比普通的 projection 多一个分支
# SwiGLU = 两个线性层并行，一个做 SiLU gate，一个做 content，然后逐元素乘，再 down projection。
## Swish (等于 silu) GLU; GLU: gated linear unit, W1x * W2 gate(x)
# x
# ├── W1 → content
# └── W2 → sigmoid → gate
#          ↓
#       elementwise multiply

# silu =  x * torch.sigmoid(x)
##--> 如果 x 很大 还是 x 放行；如果很小（负）， 压成 0；在 0 附近，压成 0.5 （soft gate，半开放）
## silu 是平滑版本的 relu + 自带 gating （relu 是硬门）

# 只有 gate 用 silu？ up 不用？
## --> 因为 gate 是【控制】信息流， 而 up 是提供内容，更像【数据本身】

# SwiGLU 的设计是：
# 👉 “线性内容 × 非线性门控”

# LU： Linear Unit
## ReLU: rectified LU 整流， 负的砍掉
## Leaky ReLU: max(0.01x, x)
## SiLU: Sigmoid LU
## GELU: Gaussian Error, 虽然已经不太“linear”了，但名字沿用



# nn.Linear(in_features=d_model, out_features=d_ff)
# 本质上 linear 在做 y = x @ W.T + b
class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff)
        self.up_proj = nn.Linear(d_model, d_ff)
        self.down_proj = nn.Linear(d_ff, d_model)


    def forward(self, x):
        gate = F.silu(self.gate_proj(x)) # (B, T, d_ff)
        up = self.up_proj(x)
        hidden = gate * up
        out = self.down_proj(hidden)
        return out


In [8]:
# 🧪 Debug
mlp = SwiGLUMLP(d_model=64, d_ff=128)
x = torch.randn(2, 8, 64)
out = mlp(x)
print("Output shape:", out.shape)  # (2, 8, 64)
print("Params:", sum(p.numel() for p in mlp.parameters()))

Output shape: torch.Size([2, 8, 64])
Params: 24896


In [9]:
# ✅ SUBMIT
from torch_judge import check
check('mlp')


🧪 Testing: SwiGLU MLP (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Parameter shapes (1.8ms)
  ✅ [2/5] Forward output shape (1.2ms)
  ✅ [3/5] Numerical correctness (3.2ms)
  ✅ [4/5] 2-D input (1.3ms)
  ✅ [5/5] Gradient flow (2.2ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (9.8ms total)
  Progress saved. Run status() to see your dashboard.

